# Stage 5 — Attach recall severity class

The recall record does **not** carry its severity class (I / II / III). That lives in the
enforcement dataset, joined on the recall number (`recall.product_res_number` =
`enforcement.recall_number`). Recalls predating the enforcement dataset are kept as `class n/a`,
not dropped.

Network: none — reads `snapshot/enforcement_raw.json.gz`.

In [ ]:
import gzip, json
import pandas as pd

with gzip.open("snapshot/enforcement_raw.json.gz", "rt") as f:
    enf = json.load(f)
enf_class = {e.get("recall_number"): e.get("classification") for e in enf}
print(f"{len(enf_class)} enforcement records with a classification")

links = pd.read_csv("data/recall_links.csv", dtype=str)
links["recall_class"] = links["product_res_number"].map(enf_class)

### Per-device worst class (I > II > III), corpus devices only

In [ ]:
ORDER = {"Class I": 0, "Class II": 1, "Class III": 2}
def worst(classes):
    known = [c for c in classes if c in ORDER]
    if not known:
        return "n/a"
    return sorted(known, key=lambda c: ORDER[c])[0]

corpus_links = links[links["in_corpus"] == "True"]
recalled = (corpus_links.groupby("k_number")["recall_class"]
            .apply(lambda s: worst(list(s))).reset_index()
            .rename(columns={"recall_class": "worst_class"}))
# earliest recall date per device
dates = corpus_links.groupby("k_number")["event_date_initiated"].min().reset_index()
recalled = recalled.merge(dates, on="k_number", how="left")
recalled.to_csv("data/recalled_nodes.csv", index=False)
links.to_csv("data/recall_links_classified.csv", index=False)

print(f"CHECKPOINT  recalled corpus devices {len(recalled)}")
print(f"CHECKPOINT  worst-class distribution {recalled['worst_class'].value_counts().to_dict()}")